# CSE304 X-ray — 실행 노트북 v3

실제 분석 코드는 `src/xraydm/*.py`에 있습니다. 이 노트북은 Drive 연결·설치·실행·결과 검사만 수행합니다.

**필수:** 기존 8개 KAMP ZIP + **새 `xray-labels-2026-11-10.zip` 1개**. 이전 11/03 검수 ZIP을 동시에 넣으면 중단됩니다. 전체 NG 이미지 T/F는 검사기 표시 검수이며 실제 이물질 검출 정답이 아닙니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, os
BASE = Path('/content/drive/MyDrive/CSE304_Xray')
INPUT = BASE / 'input_zips'
REPO_ZIP = BASE / 'CSE304_Xray_DomainMining_Colab_Repo_v3.zip'
OUTPUT = BASE / 'results' / 'domain_v3'
assert INPUT.is_dir(), f'Missing input directory: {INPUT}'
assert REPO_ZIP.exists(), f'Upload repository ZIP to {REPO_ZIP}'
zips=list(INPUT.glob('*.zip'))
print('ZIPs:', [f.name for f in zips])
assert len(zips)==9, f'Expected 8 KAMP + 1 review ZIP, found {len(zips)}'
review=[p for p in zips if 'xray-labels' in p.name.lower()]
assert len(review)==1 and '2026-11-10' in review[0].name, 'Use ONLY the new 2026-11-10 review ZIP'
WORK=Path('/content/CSE304_Xray_Colab_Repo')
WORK.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(REPO_ZIP) as z:
    for member in z.namelist():
        if member.endswith('/'): continue
        assert member.startswith('CSE304_Xray_Colab_Repo/') and '..' not in Path(member).parts
        target=Path('/content')/member
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(z.read(member))
print('Repo ready:',WORK)


In [ ]:
%cd /content/CSE304_Xray_Colab_Repo
%pip -q install -r requirements.txt
%pip -q install -e .


## 실행 설정

처음에는 `FAST_TEST=True`, `USE_DEEP=False`로 입력·라벨·박스 제거를 검증하고, 전체 실행 때 `FAST_TEST=False`, `USE_DEEP=True`, `USE_YOLO_ROI=True`로 설정하세요. Google Colab 메뉴에서 GPU 런타임을 켜는 것을 권장합니다.


In [ ]:
import os,sys,subprocess
FAST_TEST=True
USE_DEEP=False
USE_YOLO_ROI=False
FAST_TEST_LIMIT=72
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
cmd=[sys.executable, 'run.py', '--input',str(INPUT),'--output',str(OUTPUT),'--work','/content/xray_v3_work']
if FAST_TEST:cmd+=['--max-images',str(FAST_TEST_LIMIT)]
if not USE_DEEP:cmd+=['--no-deep']
if USE_YOLO_ROI:cmd+=['--with-yolo-roi']
print('Command:', ' '.join(cmd))
subprocess.run(cmd,check=True,env=os.environ.copy())


## 결과 확인

최초 QC는 `quality_control/overlay`의 원본·마스크·보간 이미지, `audit/label_audit.json`, `audit/groups/group_label_conflicts.csv`를 직접 보면서 진행하세요. FAST_TEST에서는 검수 행 전체와 일부 원본만 불러오므로 미매칭 감사 항목이 많을 수 있습니다. 전체 실행에서만 전체 매칭이 기대됩니다.


In [ ]:
from pathlib import Path
import pandas as pd,json
print(json.loads((OUTPUT/'audit/label_audit.json').read_text()))
for rel in ['audit/groups/group_label_conflicts.csv',
            'analysis/e4/e4_group_bootstrap_statistics.csv',
            'analysis/ablations/all_reviewed/feature_family_ablation.csv',
            'analysis/ablations/all_reviewed/annotation_coverage_by_label.csv']:
    p=OUTPUT/rel
    print('\n',rel,'exists:',p.exists())
    if p.exists():display(pd.read_csv(p).head(12))
from IPython.display import display,Image
pictures=sorted((OUTPUT/'quality_control/overlay').glob('overlay_*.png'))
if pictures:display(Image(filename=str(pictures[0]),width=1100))


## 해석 한계

- 본 프로젝트는 NG 영상의 **표시 적절성** 연구입니다.
- F 중에는 다양한 예외가 섞여 있고, 지원 박스 0개라는 이유만으로 프레임 이상이라고 확정하지 않습니다.
- 인페인팅은 원본 X-ray를 복구하지 않습니다.
- 후보별 covered/missed의 검증된 정답이 없으므로 후보 수준 미검출률을 보고하지 않습니다.
- 전체 GPU 학습 결과를 생성하기 전까지 실제 AP와 p-value를 발표 자료에 기입하지 마세요.
